# Signed Jumps 03 — do "bubbles" (aggressor activity) predict when a jump happens?

Protocol: `discussions/Signed_Jumps/005_claude_sj03_preregistration.md` (fixed before this run).

Same jumps as SJ 01. Models (b) → (bf) and (e) → (f) add order-flow heat readings (trade/volume acceleration, |taker imbalance|).
Gates: **F1** (f) > (e); **F2** 15-min forecast (f) > (e); **F3** (f) > (e + flow-implied volatility) — the only gate that shows boil-specific heat.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_flow as sf
for m in (sj,sf):importlib.reload(m)
assert sj.VERSION.startswith('SJ01-0.1') and sf.VERSION=='SJ03-0.1.0'
SYNTHETIC=os.getenv('SJ03_SYNTHETIC','0')=='1'
p=sj.Params();fp=sf.FlowParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj03-');OUT=Path(tmp.name)/'03'
    p=sj.Params(n_boot=200,min_test_jumps_per_sign=20,mc_paths=100);fp=sf.FlowParams(n_boot=200)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'03'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_flow.py'),str(ROOT/'tests'/'test_signed_jumps.py')],
                       cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(sf.registration(fp),indent=1))

.................                                                        [100%]
17 passed in 74.93s (0:01:14)


{
 "accel_pairs": [
  [
   10,
   240
  ],
  [
   60,
   1440
  ]
 ],
 "imb_window": 10,
 "min_cover": 0.75,
 "sj01_half_life": 1.0,
 "profile_window": [
  -30,
  10
 ],
 "n_boot": 2000,
 "status": "PREREGISTERED_SJ005",
 "version": "SJ03-0.1.0",
 "gates": {
  "F1": "(f) vs (e) TEST logLik/row delta > 0 and day-bootstrap q2.5 > 0, >= 200 TEST jumps/sign",
  "F2": "15-min signed occurrence log-loss (f) vs (e): delta > 0 and q2.5 > 0",
  "F3": "(f) vs (e + flow-implied forward variance v_hat): same rule as F1"
 }
}


## Data: SJ 01 engine (unchanged) + order flow

In [2]:
T0=time.time()
if SYNTHETIC:
    S0,F0=sj.simulate(n_days=60,n_assets=len(p.assets),eta=((0.2,0.05),(0.05,0.3)),half_life=1,seed=12,df=8,base_rate=8e-4,
                      flow=True,heat_link=1.0,flow_heat=1.0,flow_vol=0.5)
    S={a:s for a,s in zip(p.assets,S0.values())};FR={a:f for a,f in zip(p.assets,F0.values())}
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
assets,pan,dev_end=sj.prepare(S,p)
if SYNTHETIC:
    flows={a.name:FR[a.name].reindex(a.time) for a in assets}
else:
    flows={a.name:sf.load_spot_flow(BIN/'basis'/'spot'/f'{a.name}USDT_spot_1m.parquet',a.time) for a in assets}
    q01=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_data_quality.csv').set_index('asset')
    for i,a in enumerate(assets):
        m=pan.asset_idx==i
        for s in sj.SIGNS:
            for per,mm in (('dev',pan.dev),('test',~pan.dev)):
                assert int(pan.J[s][m&mm].sum())==int(q01.loc[a.name,f'{s}_{per}']),(a.name,s,per)
    print('jump counts reproduce SJ 01')
pf=sf.attach_flow(assets,pan,flows,fp)
cov=pd.DataFrame({'asset':[a.name for a in assets],'flow_missing_share':[float(flows[a.name].isna().any(axis=1).mean()) for a in assets],
                  'panel_rows_sj01':[int((pan.asset_idx==i).sum()) for i in range(len(assets))],
                  'panel_rows_flow':[int((pf.asset_idx==i).sum()) for i in range(len(assets))]})
display(cov);cov.to_csv(OUT/'03_coverage.csv',index=False)
print('flow covariates:',pf.extra_names,'| DEV scaler',pf.extra_scaler,f'| {time.time()-T0:.0f}s')

D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:172: RuntimeWarning: divide by zero encountered in log
  feats.append(np.log(m))
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: divide by zero encountered in divide
  Z = z / np.sqrt(sig2c)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: invalid value encountered in divide
  Z = z / np.sqrt(sig2c)


jump counts reproduce SJ 01


,asset,flow_missing_share,panel_rows_sj01,panel_rows_flow
0,BTC,0.0,1156376,1156376
1,ETH,0.0,1156334,1156334
2,SOL,0.0,1156468,1156468
3,XRP,0.0,1156364,1156364


flow covariates: ('AN10', 'AV10', 'AN60', 'AV60', 'IMB10') | DEV scaler {'mean': [-0.09455041399676668, -0.16854520180228436, -0.13373520627118501, -0.17548397965702195, 0.1865174517063247], 'sd': [0.47998120839414765, 0.6094459155187609, 0.5432959650932969, 0.6100307673112181, 0.14099383095786544]} | 92s


## Fit (b), (bf), (e), (f) on DEV; F1 on TEST

In [3]:
T1=time.time()
fits,E=sf.fit_models(assets,pf,p,fp)
print(f'fits {time.time()-T1:.0f}s; converged:',{k:f.converged for k,f in fits.items()})
cmp,nj=sf.compare(fits,pf,E,p)
print('TEST jumps',nj);display(cmp[['test','delta_per_row','q025','q975','n_days','status']])
dev=pf.dev;se_rows=[]
for m in ('bf','f'):
    X=sj.design(pf,m)[dev].astype(float)
    for s in sj.SIGNS:
        f=fits[(m,s)];Ex=sj.excitation_design(E[fp.sj01_half_life],m,s);Ex=None if Ex is None else Ex[dev]
        se=sj.cluster_se(f,X,Ex,pf.J[s][dev],pf.day[dev])
        names=sj.design_names(pf,m,p)+(['eta_from_up','eta_from_down'] if Ex is not None else [])
        for n,v,e in zip(names,np.concatenate([f.b,f.eta]),se):se_rows.append({'model':m,'child_sign':s,'param':n,'estimate':v,'cluster_se':e})
se_tab=pd.DataFrame(se_rows);se_tab['t']=se_tab.estimate/se_tab.cluster_se;se_tab.to_csv(OUT/'03_param_se.csv',index=False)
display(se_tab[se_tab.param.isin(list(pf.extra_names)+['TV10','TV60'])].round(4))

fits 105s; converged: {('b', 'up'): True, ('b', 'down'): True, ('bf', 'up'): True, ('bf', 'down'): True, ('e', 'up'): True, ('e', 'down'): True, ('f', 'up'): True, ('f', 'down'): True}
TEST jumps {'up': 3546, 'down': 3212}


,test,delta_per_row,q025,q975,n_days,status
0,F1_f_vs_e,0.000147,0.000078,0.000215,242,PASS
1,bf_vs_b,0.000412,0.000298,0.000536,242,PASS
2,e_vs_b,0.000945,0.000761,0.001136,242,PASS
3,f_vs_b,0.001092,0.000899,0.001288,242,PASS


,model,child_sign,param,estimate,cluster_se,t
15,bf,up,TV10,0.3271,0.0407,8.0302
16,bf,up,TV60,-0.3171,0.0912,-3.4780
17,bf,up,AN10,0.2159,0.0326,6.6129
18,bf,up,AV10,0.4514,0.0296,15.2569
19,bf,up,AN60,0.0009,0.0506,0.0170
20,bf,up,AV60,0.1413,0.0429,3.2904
21,bf,up,IMB10,0.0858,0.0141,6.0929
37,bf,down,TV10,0.3679,0.0446,8.2500
38,bf,down,TV60,-0.2460,0.1086,-2.2655
39,bf,down,AN10,0.3512,0.0361,9.7270


## F3: is it boil-specific, or just a sharper volatility thermometer?

In [4]:
vp,vinfo=sf.vhat_panel(assets,pf,p)
print(json.dumps(vinfo,indent=1))
f3,fits_v=sf.compare_f3(fits,E,vp,assets,p,pf)
cmp=pd.concat([cmp,pd.DataFrame([f3])],ignore_index=True);cmp.to_csv(OUT/'03_tests.csv',index=False)
display(cmp[['test','delta_per_row','q025','q975','status']])

{
 "flow_beta_on_fwd_var": {
  "AN10": 0.0843,
  "AV10": 0.0432,
  "AN60": 0.0645,
  "AV60": -0.0129,
  "IMB10": 0.0039
 },
 "fwd_var_R2_base_dev": 0.04006565391354888,
 "fwd_var_R2_with_flow_dev": 0.06037199014599248
}


,test,delta_per_row,q025,q975,status
0,F1_f_vs_e,0.000147,0.000078,0.000215,PASS
1,bf_vs_b,0.000412,0.000298,0.000536,PASS
2,e_vs_b,0.000945,0.000761,0.001136,PASS
3,f_vs_b,0.001092,0.000899,0.001288,PASS
4,F3_f_vs_ev,-0.000006,-0.000025,0.000012,FAIL


## F2: 15-minute jump forecast; lift table

In [5]:
T2=time.time()
hz,pr,ro=sf.horizon_compare(fits,assets,pf,p)
print(f'{time.time()-T2:.0f}s');display(hz[['test','horizon_min','delta_per_row','q025','q975','status']]);hz.to_csv(OUT/'03_horizon.csv',index=False)
lift=sf.lift_table(pr,ro,pf,p.primary_horizon);lift.to_csv(OUT/'03_lift_15min.csv',index=False)
display(lift.pivot_table(index='dec',columns='model',values=['realised','lift'],aggfunc='first').round(4))

244s


,test,horizon_min,delta_per_row,q025,q975,status
0,F2_f_vs_e,5,0.000296,-0.000174,0.000788,FAIL
1,bf_vs_b,5,0.000119,-0.000550,0.000819,FAIL
2,F2_f_vs_e,15,-0.001250,-0.002237,-0.000265,FAIL
3,bf_vs_b,15,-0.003670,-0.005089,-0.002216,FAIL
4,F2_f_vs_e,60,-0.013669,-0.016525,-0.010666,FAIL
5,bf_vs_b,60,-0.025901,-0.030276,-0.021504,FAIL


lift                 realised                
model       b       e       f        b       e       f
dec                                                   
0      0.8447  0.8237  0.7676   0.0519  0.0506  0.0471
1      0.8114  0.8062  0.7781   0.0498  0.0495  0.0478
2      0.8219  0.7694  0.7518   0.0505  0.0473  0.0462
3      0.8114  0.8272  0.8097   0.0498  0.0508  0.0497
4      0.8885  0.8815  0.8289   0.0546  0.0541  0.0509
5      0.9517  0.9149  0.9395   0.0584  0.0562  0.0577
6      0.9726  0.9586  0.9726   0.0597  0.0589  0.0597
7      1.0445  1.0112  1.1269   0.0641  0.0621  0.0692
8      1.1900  1.2566  1.1865   0.0731  0.0772  0.0729
9      1.6631  1.7508  1.8384   0.1021  0.1075  0.1129
top1%  2.6262  2.8013  3.2740   0.1613  0.1720  0.2011

## Event-time profile: is there a build-up of activity before jumps? (descriptive)

In [6]:
prof=sf.event_profile(assets,pf,flows,fp);prof.to_csv(OUT/'03_event_profile.csv',index=False)
display(prof.pivot_table(index='k',columns=['series'],values='excess',aggfunc='mean').loc[[-30,-20,-10,-5,-3,-2,-1,0,1,2,5,10]].round(3))

series,quote_volume,trades
k,,
-30,0.055,0.077
-20,0.000,0.004
-10,0.116,0.103
-5,0.242,0.193
-3,0.231,0.196
-2,0.342,0.275
-1,0.555,0.457
0,2.396,1.861
1,1.578,1.396


## Decision

In [7]:
dec=sf.decide(cmp,hz,f3,p)
dec.update(flow_vhat_info=vinfo,test_jumps=nj,dev_end=str(dev_end),synthetic=SYNTHETIC,
           top1pct_15min={m:lift[(lift.model==m)&(lift.dec.astype(str)=='top1%')][['realised','lift']].iloc[0].to_dict() for m in ('b','e','f')})
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'03_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'03_manifest.json').write_text(json.dumps({'registration':sf.registration(fp),'runtime_s':time.time()-T0,'rows':int(len(pf.r))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

{
 "version": "SJ03-0.1.0",
 "F1_flow_adds_to_SJ01": "PASS",
 "F2_15min_forecast": "FAIL",
 "F3_jump_specific_heat": "FAIL",
 "flow_vs_engine_only_bf_vs_b": "PASS",
 "flow_vhat_info": {
  "flow_beta_on_fwd_var": {
   "AN10": 0.0843,
   "AV10": 0.0432,
   "AN60": 0.0645,
   "AV60": -0.0129,
   "IMB10": 0.0039
  },
  "fwd_var_R2_base_dev": 0.04006565391354888,
  "fwd_var_R2_with_flow_dev": 0.06037199014599248
 },
 "test_jumps": {
  "up": 3546,
  "down": 3212
 },
 "dev_end": "2026-02-15 00:00:00+00:00",
 "synthetic": false,
 "top1pct_15min": {
  "b": {
   "realised": 0.16129032258064516,
   "lift": 2.6262394988862883
  },
  "e": {
   "realised": 0.17204301075268819,
   "lift": 2.8013221321453745
  },
  "f": {
   "realised": 0.2010752688172043,
   "lift": 3.274045241944906
  }
 }
}
total 544s
